# Batch covariate resolution on integration

In [ ]:
import scanpy as sc
import rapids_singlecell as rsc
from matplotlib import pyplot as plt
import seaborn as sns
import pandas as pd
from tqdm import tqdm
from pprint import pprint

sc.set_figure_params(frameon=False, dpi=80, fontsize=12)

from scatlastb_utils.io import read_anndata
from scatlastb_utils.pipeline.misc import dask_compute

from batch_analysis_utils import clean_slot_names

# Read Data

In [ ]:
adata = read_anndata(
    'data/pipeline/batch_analysis/majority_voting/dataset~batch_correction/file_id~collect:batch_correction.zarr/',
    X='X',
    obs='obs',
    var='var',
    obsm='obsm',
    uns='uns',
    obsp='obsp',
    dask_slots=['layers', 'X', 'obsp'],
    dask=True,
    backed=True,
)

# Marker genes

In [ ]:
from marker_genes_functions import filter_marker_genes, extract_de_genes, update_marker_gene_keys

In [ ]:
adata.var_names = adata.var['feature_name'].astype(str)

In [ ]:
markers, gene_set = filter_marker_genes('configs/HLCA_gene_sets.yaml', adata)

In [ ]:
genes_to_plot = extract_de_genes(adata) | gene_set

In [ ]:
%%time
# subset to genes
adata = dask_compute(adata[:, adata.var_names.isin(genes_to_plot)].copy())

In [ ]:
clean_slot_names(
    adata,
    replace_str='metrics:marker_genes:label_transfer:clustering:integration:preprocessing:'
)

In [ ]:
# set colors
ref_key = 'ann_finest_level'
for col in tqdm([x for x in adata.obs.columns if x.startswith('majority_reference')]):
    adata.obs[col] = adata.obs[col].cat.set_categories(adata.obs[ref_key].cat.categories)

## Pick integration

In [ ]:
integration1 = 'scanvi_dataset'
integration2 = 'scanvi_sample'

In [ ]:
for col in ['ann_finest_level', 'dataset']:
    sc.pl.embedding(
        adata,
        basis=f'X_umap--{integration1}',
        color=col,
        ncols=1
    )

    sc.pl.embedding(
        adata,
        basis=f'X_umap--{integration2}',
        color=col,
        ncols=1
    )

# Graph dissimilarity

In [ ]:
from scatlastb_utils.metrics import compare_distances, plot_distances_scatter, plot_ranked_distances

In [ ]:
pprint(list(adata.obsp.keys()))

In [ ]:
%%time
comp = f'{integration1}-vs-{integration2}'

adata.obs[[
    f'avg_distance_1:{comp}',
    f'avg_distance_2:{comp}',
    f'avg_difference:{comp}',
    f'avg_distance_diff:{comp}',
    f'spearman_correlation:{comp}',
]] = compare_distances(
    adata,
    obsp_key_1=f'distances--{integration1}',
    obsm_key_1=f'X_emb--{integration1}',
    obsp_key_2=f'distances--{integration2}',
    obsm_key_2=f'X_emb--{integration2}',
    # k_max=100,
    scale_distances=True,
    log_scale_diffs=False,
)

comp = f'{integration2}-vs-{integration1}'
adata.obs[[
    f'avg_distance_1:{comp}',
    f'avg_distance_2:{comp}',
    f'avg_difference:{comp}',
    f'avg_distance_diff:{comp}',
    f'spearman_correlation:{comp}',
]] = compare_distances(
    adata,
    obsp_key_1=f'distances--{integration2}',
    obsm_key_1=f'X_emb--{integration2}',
    obsp_key_2=f'distances--{integration1}',
    obsm_key_2=f'X_emb--{integration1}',
    # k_max=100,
    scale_distances=True,
    log_scale_diffs=False,
)

In [ ]:
plot_ranked_distances(adata, integration1, integration2)
plot_distances_scatter(adata, integration1, integration2, s=1, alpha=0.5)
plot_distances_scatter(adata, integration2, integration1, s=1, alpha=0.5)

In [ ]:
comparisons = [
    f'{integration1}-vs-{integration2}',
    f'{integration2}-vs-{integration1}'
]
metrics = [
    # 'avg_distance_1',
    # 'avg_distance_2',
    # 'avg_difference',
    'avg_distance_diff',
    # 'spearman_correlation'
]

In [ ]:
for _metric in metrics:
    colors = [f'{_metric}:{comparison}' for comparison in comparisons]
    sc.pl.embedding(
        adata,
        basis=f'X_umap--{integration1}',
        color=colors,
        cmap='viridis',
        size=1,
        ncols=len(metrics),
        wspace=0.5,
        # vmin=0,
        vmax=adata.obs[colors].quantile(0.99).tolist(),
        sort_order=True,
    )
 
    sc.pl.embedding(
        adata,
        basis=f'X_umap--{integration2}',
        color=colors,
        cmap='viridis',
        size=1,
        ncols=len(metrics),
        wspace=0.5,
        # vmin=0,
        vmax=adata.obs[colors].quantile(0.99).tolist(),
        sort_order=True,
    )

In [ ]:
for color in ['ann_finest_level', 'ann_level_3','ann_level_2']:
    sc.pl.embedding(
        adata,
        basis=f'X_umap--{integration1}',
        color=color,
    )
    
    sc.pl.embedding(
        adata,
        basis=f'X_umap--{integration2}',
        color=color,
    )

## Determine clusters of interest

In [ ]:
def plot_diff_per_clusters(
    adata,
    cluster_key,
    diff_key,
    x_label=None,
    width=6,
    height_factor=0.2,
    figsize=None,
    agg="mean",
    quantile=None,
    cluster_thresh=10,
):
    if x_label is None:
        x_label = diff_key

    # ensure categorical for consistent color mapping
    if not pd.api.types.is_categorical_dtype(adata.obs[cluster_key]):
        adata.obs[cluster_key] = adata.obs[cluster_key].astype("category")

    # filter clusters by size
    value_counts = adata.obs[cluster_key].value_counts()
    value_counts = value_counts[value_counts > cluster_thresh]
    clusters = value_counts.index

    df = adata.obs[adata.obs[cluster_key].isin(clusters)]
    gb = df.groupby(cluster_key, observed=True)[diff_key]

    if quantile is not None:
        diff_per_cluster = gb.quantile(quantile)
    else:
        diff_per_cluster = gb.agg(agg)

    diff_per_cluster = diff_per_cluster.sort_values()

    # ---- color mapping from AnnData ----
    colors = None
    color_key = f"{cluster_key}_colors"
    if color_key in adata.uns:
        cat_to_color = dict(
            zip(adata.obs[cluster_key].cat.categories, adata.uns[color_key])
        )
        colors = [cat_to_color.get(k, "#333333") for k in diff_per_cluster.index]

    # ---- plot ----
    if figsize is None:
        figsize = (width, len(diff_per_cluster) * height_factor)

    fig, ax = plt.subplots(figsize=figsize)

    diff_per_cluster.plot.barh(ax=ax, color=colors)

    for i, (k, v) in enumerate(diff_per_cluster.items()):
        ax.text(v, i, f"{value_counts[k]}", va="center")

    ax.grid(False)
    ax.set_xlabel(x_label)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    plt.show()

    return diff_per_cluster


In [ ]:
plot_diff_per_clusters(
    adata,
    'ann_level_1',
    f'avg_distance_diff:{integration2}-vs-{integration1}',
)

In [ ]:
color = 'ann_level_1'
sc.pl.embedding(
    adata,
    basis=f'X_umap--{integration1}',
    color=color,
)

sc.pl.embedding(
    adata,
    basis=f'X_umap--{integration2}',
    color=color,
)

In [ ]:
plot_diff_per_clusters(
    adata,
    'ann_level_2',
    f'avg_distance_diff:{integration2}-vs-{integration1}',
)

In [ ]:
plot_diff_per_clusters(
    adata,
    f'leiden_0.5_1--{integration1}',
    f'avg_distance_diff:{integration1}-vs-{integration2}',
)
plot_diff_per_clusters(
    adata,
    f'leiden_0.5_1--{integration2}',
    f'avg_distance_diff:{integration1}-vs-{integration2}',
)

In [ ]:
for col in [
    'ann_finest_level',
    # f'majority_reference--{integration1}',
    # f'majority_reference--{integration2}',
    f'leiden_0.5_1--{integration1}',
    f'leiden_0.5_1--{integration2}',
]:
    sc.pl.embedding(
        adata,
        basis=f'X_umap--{integration1}',
        color=col,
        ncols=1,
    )
    
    sc.pl.embedding(
        adata,
        basis=f'X_umap--{integration2}',
        color=col,
        ncols=1,
    )

## Investigate top cell types

In [ ]:
metrics = [
    f'avg_distance_diff:{integration1}-vs-{integration2}',
    f'avg_distance_diff:{integration2}-vs-{integration1}'
]
metric = metrics[1]

colors = metrics + [
    'ann_finest_level_select',
    'anatomical_region_ccf_score',
    'tissue_level_2',
]

In [ ]:
ann_clusters = plot_diff_per_clusters(
    adata,
    'ann_finest_level',
    metric,
    cluster_thresh=50,
    quantile=0.99,
    agg='mean',
)

In [ ]:
n_top = 5

groups = ann_clusters[-n_top:][::-1].index.tolist()
mask = adata.obs['ann_finest_level'].isin(groups)

adata.obs['ann_finest_level_select'] = adata.obs.loc[mask, 'ann_finest_level'].cat.remove_unused_categories()

In [ ]:
for col in colors:
    sc.pl.embedding(
        adata,
        basis=f'X_umap--{integration1}',
        color=col,
        mask_obs=mask,
        size=5,
        ncols=1,
        # vmax=adata.obs[metric].quantile(0.99).tolist(),
        # sort_order=False,
    )

    sc.pl.embedding(
        adata,
        basis=f'X_umap--{integration2}',
        color=col,
        mask_obs=mask,
        size=5,
        ncols=1,
        # vmax=adata.obs[metric].quantile(0.99).tolist(),
        # sort_order=False,
    )

In [ ]:
adata.obs.loc[
    mask,
    [
        'ann_finest_level',
        f'leiden_0.5_1--{integration1}',
        f'leiden_0.5_1--{integration2}',
    ]
].value_counts()

### Marker genes

In [ ]:
cluster_key = 'majority_reference'
cluster_key = 'ann_finest_level'
cluster_key = f'leiden_0.5_1--{integration1}'
cluster_key_2 = f'leiden_0.5_2--{integration1}'

In [ ]:
adata.var_names_make_unique()

In [ ]:
groups

In [ ]:
value_counts = adata.obs.query('majority_consensus == @groups[0]').value_counts(cluster_key)
subgroups = value_counts[value_counts > 10].index.to_list()
subgroups

In [ ]:
dp = sc.pl.dotplot(
    adata[adata.obs[cluster_key].isin(subgroups)],
    var_names=markers['HLCA_epithelial'],
    groupby=cluster_key,
    dendrogram=False,
    return_fig=True,
)
dp.add_totals().style(dot_edge_color='black', dot_edge_lw=0.5).show()

In [ ]:
dp = sc.pl.dotplot(
    adata[adata.obs[cluster_key].isin(subgroups)],
    var_names=markers['HLCA_epithelial'],
    groupby=cluster_key_2,
    dendrogram=False,
    return_fig=True,
)
dp.add_totals().style(dot_edge_color='black', dot_edge_lw=0.5).show()

In [ ]:
dp = sc.pl.dotplot(
    adata[adata.obs[cluster_key].isin(subgroups)],
    var_names=markers['HLCA_epithelial'],
    groupby=f'majority_reference--{integration1}',
    dendrogram=False,
    return_fig=True,
)
dp.add_totals().style(dot_edge_color='black', dot_edge_lw=0.5).show()

In [ ]:
dp = sc.pl.dotplot(
    adata[adata.obs['majority_consensus'].isin(groups)],
    var_names=markers['HLCA_epithelial'],
    groupby=cluster_key,
    dendrogram=False,
    return_fig=True,
)
dp.add_totals().style(dot_edge_color='black', dot_edge_lw=0.5).show()

In [ ]:
dp = sc.pl.dotplot(
    adata[adata.obs[cluster_key].isin(subgroups)],
    var_names=markers['HLCA_stroma'],
    groupby=cluster_key_2,
    dendrogram=False,
    return_fig=True,
)
dp.add_totals().style(dot_edge_color='black', dot_edge_lw=0.5).show()

## Subset to Epithelium

In [ ]:
def recompute_knn(adata, integration, n_neighbors=100):
    rsc.pp.neighbors(adata, use_rep=f'X_emb--{integration}', n_neighbors=n_neighbors)
    rsc.tl.umap(
        adata,
        min_dist=0.5,
        spread=1.0,
        n_components=2,
        maxiter=None,
        alpha=1.0,
        # gamma=1.0,
        negative_sample_rate=5
    )
    adata.obsp[f'connectivities--{integration}'] = adata.obsp['connectivities']
    adata.obsp[f'distances--{integration}'] = adata.obsp['distances']
    adata.obsm[f'X_umap--{integration}'] = adata.obsm['X_umap']

In [ ]:
adata_epi = adata[adata.obs['ann_level_1'] == 'Epithelial'].copy()
# adata_epi = adata[adata.obs['ann_level_2'] == 'Airway epithelium'].copy()

In [ ]:
recompute_knn(adata_epi, integration1, n_neighbors=100)
recompute_knn(adata_epi, integration2, n_neighbors=100)

In [ ]:
cluster_key = 'ann_finest_level'
mask = adata_epi.obs[cluster_key].isin(groups)

In [ ]:
adata_epi.var.index.name = 'index'

In [ ]:
adata_epi.write_zarr('data/output/batch_analysis/hlca_epithelial.zarr')

In [ ]:
colors = [
    'ann_finest_level',
    f'leiden_0.5_2--{integration1}',
    f'leiden_0.5_2--{integration2}',
    'study',
    'tissue_level_2',
]

In [ ]:
sc.pl.embedding(
    adata_epi,
    basis=f'X_umap--{integration1}',
    color='ann_finest_level',
    ncols=1,
    palette=sc.pl.palettes.default_102,
    size=1,
)

sc.pl.embedding(
    adata_epi,
    basis=f'X_umap--{integration2}',
    color='ann_finest_level',
    ncols=1,
    palette=sc.pl.palettes.default_102,
    size=1,
)